In [1]:
import shutil
import time

### Copy files to avoid duplication

In [2]:
list_str_filenames = [
    'functions.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
    'api.py',
    'passwords.py',
]
for str_filename in list_str_filenames:
    str_source = f'../01_single/{str_filename}'
    str_destination = f'./{str_filename}'
    shutil.copyfile(str_source, str_destination)

# pause so we can read the recently copied modules (i.e., api.py)
time.sleep(5)

In [3]:
import os
import time
import json
import pandas as pd
import pickle
from functions import *
from pprint import pprint
import boto3
from passwords import *
# silence
pd.options.mode.chained_assignment = None

### Functions

In [4]:
# upload to s3
def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name, aws_session_token=None):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )
    # upload
    cls_client.download_file(
        str_project, 
        str_bucket_key, 
        str_local_path,
    )

In [5]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name, aws_session_token=None):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [6]:
try:
    str_project = os.getcwd().split('/')[4].replace('_','-')
except IndexError:
    str_project = os.getcwd().split('\\')[4].replace('_','-') 
print(f'Project: {str_project}')
str_dirname_output = './output'
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii


### Output directory

In [7]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [8]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Import payload

In [9]:
str_filename = 'request_8004842_w_class_codebtor.json'
str_local_path = f'./input/{str_filename}'
# load
try:
    dict_json_request = json.load(open(str_local_path, 'r'))['request']
except KeyError:
    dict_json_request = json.load(open(str_local_path, 'r'))

### Get the preprocessing script

In [10]:
str_filename = 'preprocessing.py'
str_local_path = f'./{str_filename}'
str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_path, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)

### Load parser

In [11]:
%%time

str_filename = 'cls_parser.pkl'
str_local_path = f'../01_single/output/{str_variant}/{str_filename}'
cls_parse_payload = pickle.load(open(str_local_path, 'rb'))

Wall time: 1.69 s


### Parse payload

In [12]:
# get data
cls_parse_payload.get_data(dict_json_request)
# output
#cls_parse_payload.dict_output['dict_list_tables']

[800484299135421, 800484299135430]: Get Data: 0.17323 sec.


In [13]:
# shared preprocessing
cls_parse_payload.shared_preprocessing()

# rm preprocessing.py
os.remove('preprocessing.py')

# output
cls_parse_payload.dict_output['X_clean']

NaN Replacer: 0.0038832 sec.


100%|██████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 1494.76it/s]


Set strings: 0.0093596 sec.
Boolean Replacer: 0.0025145 sec.


100%|██████████████████████████████████████████████████████████████████████████████| 325/325 [00:00<00:00, 7383.54it/s]


Data Type Setter: 0.12643 sec.


...

100%|██████████████████████████████████████████████████████████████████████████████| 330/330 [00:00<00:00, 3882.29it/s]


Replace inf and -inf with NaN: 0.14958 sec.
Imputer: 0.0014787 sec.
Map term: 0.0006486 sec.
Map PTI: 3.35e-05 sec.


100%|██████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 1498.44it/s]


Round values: 0.010082 sec.
Preprocessing Model: 0.5003 sec.
[800484299135421, 800484299135430]: Shared Preprocessing: 0.52214 sec.


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bankruptcycount24month__ln,inquiryshortterm12month__ln,re01s__tu,bankruptcystatus__ln,bankruptcytimenewest__ln,fltgrossmonthly__income_sum,linka006__tu,...,intservicecontractmileage__app,fltapprovedservicecontract__app,fltgapinsurance__app,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
0,800484299135421,8004842,9913542,0.0,0.0,0.0,0.0,0.0,9500.0,0.0,...,120000.0,0.0,0.0,2024,0.9409,10,4,0.931034,7.0,21.265753
0,800484299135430,8004842,9913543,0.0,0.0,0.0,0.0,0.0,19000.0,0.0,...,120000.0,0.0,0.0,2024,0.9409,10,4,0.931034,7.0,21.265753


In [14]:
# predict
cls_parse_payload.generate_predictions()
# output
for str_key in ['y_hat_ad','mean_ad','y_hat_pd','y_hat_lgd','ecnl','ecnl_mod']:
    print('')
    print(str_key)
    print(cls_parse_payload.dict_output[str_key])

[800484299135421, 800484299135430]: Generate Predictions: 0.02940 sec.

y_hat_ad
0    0.37885
1    0.35325
dtype: float64

mean_ad
0.36605013936607245

y_hat_pd
0    0.199744
1    0.197792
dtype: float64

y_hat_lgd
0    0.681574
1    0.679959
dtype: float64

ecnl
0.13531482938797976

ecnl_mod
0.3193429973556322


In [15]:
# adverse action
cls_parse_payload.adverse_action()
# output
for list_reasons in cls_parse_payload.dict_output['list_list_reasons']:
    print('')
    pprint(list_reasons)

[800484299135421, 800484299135430]: Adverse Action: 0.17028 sec.

['Insufficient credit file, Length of Credit',
 'Derogatory public record',
 'Derogatory public record',
 'Insufficient credit file, Length of Credit',
 'Insufficient property value']

['Insufficient credit file, Length of Credit',
 'Derogatory public record',
 'Derogatory public record',
 'Insufficient credit file, Length of Credit',
 'Insufficient property value']


In [16]:
# counter offers
cls_parse_payload.counter_offers()
# output
cls_parse_payload.dict_output['df_counter_offers']

NaN Replacer: 0.0022357 sec.


100%|██████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 1497.97it/s]


Set strings: 0.0051793 sec.
Boolean Replacer: 0.0023612 sec.


100%|██████████████████████████████████████████████████████████████████████████████| 475/475 [00:00<00:00, 7090.67it/s]


Data Type Setter: 0.14791 sec.


...

Vehicle Class: Class 1
Vehicle Class: Class 1


100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 399.80it/s]

Counter threshold: 0.33599999999999997
Initial ECNL: 0.3193
Approved (T/F): True
Tier: C
Initial offer Approved: 0.3193
Original Values:
{'LTV': 0.9328966801515444, 'APR': 0.2355, 'Fees': 1019.0, 'Tier': 'C'}
Original LTV: 0.9329
Minimum LTV: 0.8396
Original APR: 0.2355
Original Fees: 1019.0000
APR + Fees Threshold: 200
There are 0 counters on amount financed
Looking for counters on down
There are 1 counters on down


Best Counter Offer: 3
   Offer      ECNL  Decision  DownCash  AmountFinanced  SalesPrice     APR  \
0      0  0.319343  Approved      5000           14528       17200  0.2355   
3      3  0.279127  Approved      6500           13028       17200  0.2355   

   NetDiscount  CurrentLTV    MaxLTV  
0       1019.0    0.932897  0.932897  
3        359.0    0.836576  0.932897  
0.2355
1019.0
[800484299135421, 800484299135430]: Counter Offers: 0.91423 sec.


,Offer,ECNL,Decision,DownCash,AmountFinanced,SalesPrice,APR,NetDiscount,CurrentLTV,MaxLTV
0,0,0.319343,Approved,5000,14528,17200,0.2355,1019.0,0.932897,0.932897
3,3,0.279127,Approved,6500,13028,17200,0.2355,359.0,0.836576,0.932897


### Show output

In [17]:
# generate output
cls_parse_payload.generate_output()
# output
cls_parse_payload.dict_output['output_final']

2
            Row_id  Score_ad  Score_pd  Score_lgd  Score_ecnl  Score_ecnl_mod  \
0  800484299135421   0.37885  0.199744   0.681574    0.135315        0.319343   
1  800484299135430   0.35325  0.197792   0.679959    0.135315        0.319343   

      APR  Net_discount                                        Key_factors  \
0  0.2355        1019.0  [Insufficient credit file, Length of Credit, D...   
1  0.2355        1019.0  [Insufficient credit file, Length of Credit, D...   

   Outlier_score                                         Dict_tiers  
0            0.0  \n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'...  
1            0.0  \n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'...  
[{"Row_id":800484299135421,"Score_ad":0.3788498663,"Score_pd":0.1997440792,"Score_lgd":0.6815743291,"Score_ecnl":0.1353148294,"Score_ecnl_mod":0.3193429974,"APR":0.2355,"Net_discount":1019.0,"Key_factors":["Insufficient credit file, Length of Credit","Derogatory public record","Derogatory public record","Ins

{'Request_id': '',
 'Zaml_processing_id': '',
 'Response': [{'Model_name': 'prestige-gen-xii',
   'Model_version': 'v1',
   'Results': [{'Row_id': 800484299135421,
     'Score_ad': 0.3788498663,
     'Score_pd': 0.1997440792,
     'Score_lgd': 0.6815743291,
     'Score_ecnl': 0.1353148294,
     'Score_ecnl_mod': 0.3193429974,
     'APR': 0.2355,
     'Net_discount': 1019.0,
     'Key_factors': ['Insufficient credit file, Length of Credit',
      'Derogatory public record',
      'Derogatory public record',
      'Insufficient credit file, Length of Credit',
      'Insufficient property value'],
     'Outlier_score': 0.0,
     'Dict_tiers': "\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"},
    {'Row_id': 800484299135430,
     'Score_ad': 0.3532504124,
     'Score_pd': 0.1977924209,
     'Score_lgd': 0.67995931,
     'Score_ecnl': 0.1353148294,
     'Score_ecnl_mod': 0.3193429974,
     'APR': 0.2355,
     'Net_discount': 1019.0,
     'Key_factors': ['Insuffic

### Save dictionary of output for analysis in next step

In [18]:
str_filename = 'dict_output.pkl'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
pickle.dump(cls_parse_payload.dict_output, open(str_local_path, 'wb'))

### Clean-up

In [19]:
for str_filename in list_str_filenames:
    os.remove(str_filename)